In [2]:
!pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.7 MB/s eta 0:00:00


In [3]:
import torch
import bitsandbytes as bnb
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,
    EarlyStoppingCallback
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset

1. Definición del modelo que vamos a utilizar

In [4]:
model_checkpoint = "google/gemma-2-2b-it"
n_labels = 2

2. Configuración de Cuatización a 4-bits

In [6]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("Cargando tokenizador y modelo base Gemma")

Cargando tokenizador y modelo base Gemma


3. Tokenizador

In [7]:
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

4. Inicialización del Modelo Base Cuantizado

In [8]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=n_labels,
    quantization_config=bnb_config,
    device_map="auto"
)
model.config.pad_token_id = tokenizer.pad_token_id

model.safetensors.index.json:   0%|          | 0.00/24.2k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Gemma2ForSequenceClassification LOAD REPORT from: google/gemma-2-2b-it
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


5. Buscador dinámico de capas para LoRA

In [9]:
def find_all_linear_names(model):
    cls = bnb.nn.Linear4bit
    lora_module_names = set()
    for name, module in model.named_modules():
        if isinstance(module, cls):
            names = name.split('.')
            lora_module_names.add(names[0] if len(names) == 1 else names[-1])
    if 'lm_head' in lora_module_names:
        lora_module_names.remove('lm_head')
    if 'score' in lora_module_names:
        lora_module_names.remove('score')
    return list(lora_module_names)

modules = find_all_linear_names(model)
print(f"Módulos objetivo encontrados para LoRA: {modules}")

Módulos objetivo encontrados para LoRA: ['q_proj', 'down_proj', 'up_proj', 'o_proj', 'gate_proj', 'k_proj', 'v_proj']


6. Configuración de LoRA

In [10]:
model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=modules,
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_CLS",
    modules_to_save=["score"]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 20,771,328 || all params: 2,635,117,824 || trainable%: 0.7883


7. Preparación del dataset

In [11]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [12]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/EXIST2025_training_3_1.csv")
df = df.rename(columns={"label_task_3_1_merged": "label"})
df = df.drop(columns=["Unnamed: 0"])
df["label"] = df["label"].astype(int)

# from sklearn.model_selection import train_test_split
# train_df, eval_df = train_test_split(df, test_size=0.15, stratify=df["label"], random_state=42)

# train_dataset = Dataset.from_pandas(train_df)
# eval_dataset = Dataset.from_pandas(eval_df)

# PENDIENTE DE DEFINIR UN FICHERO TRAIN, VAL Y TEST
# División estratificada
train_df, temp_df = train_test_split(df, test_size=0.3, stratify=df["label"], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label'], random_state=42)

print("Distribución fichero de entrenamiento:")
print(train_df.value_counts('label'))

print("Distribución fichero de test:")
print(test_df.value_counts('label'))

print("Distribución fichero de validación:")
print(val_df.value_counts('label'))

Distribución fichero de entrenamiento:
label
0    914
1    841
Name: count, dtype: int64
Distribución fichero de test:
label
0    196
1    181
Name: count, dtype: int64
Distribución fichero de validación:
label
0    196
1    180
Name: count, dtype: int64


In [13]:
def tokenize_data(example):
    return tokenizer(example["text"], padding="max_length", truncation=True, max_length=128)

train_dataset = Dataset.from_pandas(train_df)
valid_dataset = Dataset.from_pandas(val_df)

train_dataset.reset_format()
valid_dataset.reset_format()

columns_train = train_dataset.column_names
columns_valid = valid_dataset.column_names
columna_etiqueta = "label" if "label" in columns_train else "labels"

if columna_etiqueta in columns_train: columns_train.remove(columna_etiqueta)
if columna_etiqueta in columns_valid: columns_valid.remove(columna_etiqueta)

encoded_train_dataset = train_dataset.map(tokenize_data, batched=True, remove_columns=columns_train)
encoded_valid_dataset = valid_dataset.map(tokenize_data, batched=True, remove_columns=columns_valid)


Map:   0%|          | 0/1755 [00:00<?, ? examples/s]

Map:   0%|          | 0/376 [00:00<?, ? examples/s]

8. Hiperparámetros del entrenamiento

In [19]:
training_args = TrainingArguments(
    output_dir='/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/LLMs/Gemma_2B_QLoRA',
    num_train_epochs=2,
    learning_rate=2e-4,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=8,
    eval_strategy='steps',
    eval_steps=0.2,              # Evaluamos cada 20%
    save_strategy='steps',
    save_steps=0.2,              # OBLIGATORIO: Mismo valor que eval_steps
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    weight_decay=0.01,
    fp16=False,
    bf16=True,
    optim="paged_adamw_8bit",
    report_to="none",
    logging_strategy='steps',
    logging_steps=0.2            # Para ver los logs al mismo tiempo
)

9. Entrenamiento

In [16]:
import sklearn as sk
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score, f1_score


In [17]:
# Función para realizar distintas métricas en ejecución

def compute_metrics(eval_pred):

  ##############
  ## predictions son logits, que son tuplas de la forma [valor1, valor2]
  ## Por ejemplo [-1.5606991,  1.6122842] significa que ha predicho eso para un documento
  ## Eso es lo que pasa a la última capa del transformer (softmax si es binario)
  ## Por eso se utiliza el índice del valor máximo de la tupla, para decir que esa es la clase que predice

  ## label_ids = [0, 1, 1, 0, 1]  # Etiquetas reales
  ## predictions = [
  ##  [0.8, 0.2],  # Predicciones para la primera instancia
  ##  [0.3, 0.7],  # Predicciones para la segunda instancia
  ##  [0.1, 0.9],  # Predicciones para la tercera instancia
  ##  [0.9, 0.1],  # Predicciones para la cuarta instancia
  ##  [0.4, 0.6],  # Predicciones para la quinta instancia
  ##           ]

  ##############

  labels = eval_pred.label_ids
  preds = eval_pred.predictions.argmax(-1)

  # Compute precision, recall, F1-score, and support
  precision, recall, f1, _ = sk.metrics.precision_recall_fscore_support(labels, preds, average="macro")

  # Calculate F1-score for the minority class (label = 1)
  f1_minoritaria= f1_score(labels, preds, pos_label=1)

  # Calculate F1-score for the majority class (label = 0)
  f1_mayoritaria = f1_score(labels, preds, pos_label=0)

  # Calculate accuracy
  acc = sk.metrics.accuracy_score(labels, preds)

  # Calculate Area Under the Curve (AUC)
  AUC = roc_auc_score(labels, preds)

  # Calculate Precision-Recall Area Under the Curve (AUC)
  PREC_REC = average_precision_score(labels, preds)

  return {
      'accuracy': acc,
      'f1': f1,
      'precision': precision,
      'recall': recall,
      'AUC': AUC,
      'f1_minoritaria': f1_minoritaria,
      'f1_mayoritaria': f1_mayoritaria,
      'PREC_REC': PREC_REC
  }

In [20]:
trainer = Trainer(
    model=model,
    args=training_args,
    compute_metrics=compute_metrics, # ¡Asegúrate de tener esta función cargada en memoria!
    train_dataset=encoded_train_dataset,
    eval_dataset=encoded_valid_dataset,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("🚀 Iniciando entrenamiento de Gemma-2-2B...")
trainer.train()

🚀 Iniciando entrenamiento de Gemma-2-2B...


[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Auc,F1 Minoritaria,F1 Mayoritaria,Prec Rec
44,4.383856,1.067195,0.507979,0.432659,0.560200,0.524433,0.524433,0.639376,0.225941,0.491269
88,3.260093,0.614782,0.662234,0.653104,0.669725,0.656519,0.656519,0.596825,0.709382,0.592345
132,2.349726,0.587233,0.699468,0.698339,0.699038,0.698129,0.698129,0.679887,0.716792,0.622002
176,1.478773,0.627131,0.726064,0.724189,0.726856,0.723866,0.723866,0.701449,0.746929,0.649878
220,1.284422,0.628578,0.702128,0.701104,0.701662,0.700907,0.700907,0.683616,0.718593,0.624380


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

TrainOutput(global_step=220, training_loss=2.551373897899281, metrics={'train_runtime': 4052.5716, 'train_samples_per_second': 0.866, 'train_steps_per_second': 0.054, 'total_flos': 5513457655480320.0, 'train_loss': 2.551373897899281, 'epoch': 2.0})